# Spend Categorization

## Prompt Optimizer and Evaluation
The batch inference above works at scale, but the prompt engineering is a major challenge - we have a large hierarchy with information, and no ground truth. MLflow allows us to do some work with prompt optimization and guidance. This notebook bootstraps that process.

Tested on Serverless v4.

In [0]:
%pip install . databricks-agents dspy --upgrade
%restart_python

In [0]:
from src.utils import get_spark
from src.config import load_config

spark = get_spark()
config = load_config()

In [0]:
# Widgets for SQL
dbutils.widgets.removeAll()
dbutils.widgets.text("llm_endpoint", config.large_llm_endpoint)
dbutils.widgets.text("catalog", config.catalog)
dbutils.widgets.text("schema", config.schema_name)
dbutils.widgets.text("invoices_table", config.invoices)
dbutils.widgets.text("cat_bootstrap_table", config.cat_bootstrap)
dbutils.widgets.text("prompts_table", config.prompts)
dbutils.widgets.text("categories_table", config.categories_table)
dbutils.widgets.text("categories_str_table", config.categories_str)
dbutils.widgets.text("batch_size", "1000")
dbutils.widgets.text("max_batches", "100")
dbutils.widgets.text("cat_optimize_table", "cat_optimize")

# Get widget values for use in SQL
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
cat_bootstrap_table = dbutils.widgets.get("cat_bootstrap_table")
cat_optimize_table = dbutils.widgets.get("cat_optimize_table")
categories_str_table = dbutils.widgets.get("categories_str_table")
prompts_table = dbutils.widgets.get("prompts_table")
invoices_table = dbutils.widgets.get("invoices_table")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
batch_size = int(dbutils.widgets.get("batch_size"))
max_batches = int(dbutils.widgets.get("max_batches"))

In [0]:
LARGE_LLM = config.large_llm_endpoint
EXPERIMENT = "/Workspace/Users/scott.mckean@databricks.com/experiments/spend"

In [0]:
import mlflow
prompt_uri = f"{catalog}.{schema}.combined_invoice_prompt"
loaded_prompt = mlflow.genai.load_prompt(name_or_uri=prompt_uri)

In [0]:
import mlflow
import openai
from databricks.sdk import WorkspaceClient

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT)     

mlflow.openai.autolog()         

w = WorkspaceClient()
user_name = w.current_user.me().user_name
user_id = w.current_user.me().id
openai_client = w.serving_endpoints.get_open_ai_client()
experiment = w.experiments.get_by_name(EXPERIMENT).experiment

In [0]:
corrections = spark.sql(f"""
    SELECT
      c.order_id,
      c.guidelines,
      CONCAT(
          'Correct Categories:\n',
          'Level 1: ', c.correct_level_1, '\n',
          'Level 2: ', c.correct_level_2
      ) as expectations,
      CONCAT(
          '\nOrder ID:', i.order_id,
          '\nCost Centre:', i.cost_centre,
          '\nUnit Price:', i.unit_price,
          '\nSupplier:', i.supplier,
          '\nDescription:', i.description
      ) as invoice
    FROM {catalog}.{schema}.corrections c
    LEFT JOIN {catalog}.{schema}.invoices i
    ON c.order_id = i.order_id
""").toPandas()

In [0]:
# get our latest categories
categories = spark.sql(f"""
    SELECT categories_str
    FROM {catalog}.{schema}.categories_str
    ORDER BY datetime DESC
    LIMIT 1
""").first()['categories_str']

categories[0:500]

# get a single sample of train data
train_data = corrections.sample(50)

In [0]:
# setup scorers
from mlflow.genai.scorers import Correctness, ExpectationsGuidelines
correctness_judge = Correctness(name='correctness')
guideline_judge = ExpectationsGuidelines(name='expectations_guidelines')

In [0]:
data = [
    {
        "inputs": {'invoice':row.invoice, 'categories':categories},
        "expectations": {
            'expected_facts': [row.expectations],
            'guidelines': [row.guidelines]
        }
    }
    for _, row in train_data.iterrows()
]

In [0]:
def predict_fn(invoice: str, categories:str):
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_uri}@latest")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "user", "content": prompt.format(
            invoice=invoice,
            categories=categories)}],
    )
    return completion.choices[0].message.content

In [0]:
from mlflow.genai.optimize import GepaPromptOptimizer

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=data,
    prompt_uris=[prompt_uri],
    optimizer=GepaPromptOptimizer(
        reflection_model="databricks:/databricks-claude-sonnet-4-5",
        max_metric_calls=500),
    scorers=[correctness_judge, guideline_judge],
)

In [0]:
result.optimized_prompts

In [0]:
import pprint
optimized_prompt = result.optimized_prompts[0]
pprint.pprint(optimized_prompt.template)

## Register optimized prompt


In [0]:
from datetime import datetime
import mlflow

# Register the combined prompt in the MLflow Prompt Registry
opt_prompt_uri = f"{catalog}.{schema}.optimized_prompt"
registered_prompt = mlflow.genai.register_prompt(
    name=opt_prompt_uri,
    template=optimized_prompt.template if hasattr(optimized_prompt, 'template') else str(optimized_prompt),
    commit_message="Optimized prompt",
    tags={
        "source": "optimized_classification",
        "type": "optimized",
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
)

mlflow.genai.set_prompt_alias(
    name=opt_prompt_uri,
    alias="latest",
    version=registered_prompt.version
)

print(f"Registered {registered_prompt.name}, version {registered_prompt.version}, alias 'latest'")

## Redo Classification

In [0]:
optimized_prompt.format(invoice='', categories='')

In [0]:
import pyspark.sql.functions as F

(
  spark.createDataFrame([
    (optimized_prompt.format(invoice='', categories=''),)
    ], ["prompt"])
  .withColumn("datetime", F.current_timestamp())
  .select('datetime','prompt')
  .write.format('delta')
  .mode('append')
  .saveAsTable(config.full_prompts_table_path)
)

In [0]:
%sql
-- Initialize cat_bootstrap table with schema
-- We don't have actuals in this case, so are only storing the predictions
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :schema || '.' || :cat_optimize_table) (
  order_id STRING,
  invoice_info STRING,
  pred_level_1 STRING,
  pred_level_2 STRING,
  confidence DOUBLE,
  rationale STRING,
  classified_at TIMESTAMP,
  source STRING
) USING DELTA

In [0]:
# Get batch configuration
total_rows = spark.table(config.full_invoices_table_path).count()
print(f"Total rows: {total_rows}")
print(f"Batch size: {batch_size}")
print(f"Max batches: {max_batches}")
print(f"Target table: {catalog}.{schema}.{cat_optimize_table}")

In [0]:
spark.table(f"{catalog}.{schema}.{cat_optimize_table}").count()

In [0]:
batch_num = 0
while batch_num < int(max_batches):
    processed_count = spark.table(f"{catalog}.{schema}.{cat_optimize_table}").count()
    total_count = spark.table(config.full_invoices_table_path).count()
    
    if processed_count >= total_count:
        print(f"🎉 All {total_count} rows already processed!")
        break

    spark.sql(f"""
        MERGE INTO {catalog}.{schema}.{cat_optimize_table} AS target
        USING (
            WITH latest_prompt AS (
                SELECT prompt
                FROM {catalog}.{schema}.{prompts_table}
                ORDER BY datetime DESC
                LIMIT 1
            ),
            latest_categories AS (
                SELECT categories_str
                FROM {catalog}.{schema}.{categories_str_table}
                ORDER BY datetime DESC
                LIMIT 1
            )
            SELECT
                order_id,
                invoice_info,
                ai_result.category_level_1 AS pred_level_1,
                ai_result.category_level_2 AS pred_level_2,
                ai_result.confidence AS confidence,
                ai_result.rationale AS rationale,
                current_timestamp() AS classified_at,
                'bootstrap' AS source
            FROM (
                SELECT
                    i.*,
                    CONCAT(
                        '\nOrder ID:', ifnull(i.order_id, ''),
                        '\nCost Centre:', ifnull(i.cost_centre, ''),
                        '\nUnit Price:', ifnull(i.unit_price, ''),
                        '\nSupplier:', ifnull(i.supplier, ''),
                        '\nDescription:', ifnull(i.description, '')
                    ) as invoice_info,
                    from_json(AI_QUERY(
                        '{llm_endpoint}',
                        CONCAT(
                            p.prompt, 
                            '\\nCategories:\\n', 
                            c.categories_str, 
                            '\\nInvoice:\\n', 
                            '\nOrder ID:', ifnull(i.order_id, ''),
                            '\nCost Centre:', ifnull(i.cost_centre, ''),
                            '\nUnit Price:', ifnull(i.unit_price, ''),
                            '\nSupplier:', ifnull(i.supplier, ''),
                            '\nDescription:', ifnull(i.description, '')
                        ),
                        responseFormat => 'STRUCT<result:STRUCT<category_level_1:STRING, category_level_2:STRING, confidence:DOUBLE, rationale:STRING>>'
                    ), 'STRUCT<category_level_1:STRING, category_level_2:STRING, confidence:DOUBLE, rationale:STRING>'
                    ) AS ai_result
                FROM {catalog}.{schema}.{invoices_table} i
                CROSS JOIN latest_prompt p
                CROSS JOIN latest_categories c
                LEFT ANTI JOIN {catalog}.{schema}.{cat_optimize_table} cat
                    ON i.order_id = cat.order_id
                LIMIT {batch_size}
            )
        ) AS source
        ON target.order_id = source.order_id
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    new_processed_count = spark.table(f"{catalog}.{schema}.{cat_optimize_table}").count()

    if new_processed_count == processed_count:
        print(f"🎉 All {total_count} rows already processed!")
        break

    print(f"✅ Batch {batch_num + 1} complete. Processed: {new_processed_count} / {total_count} rows ({new_processed_count/total_count*100:.1f}%)")
    
    batch_num += 1

In [0]:
from sklearn.metrics import accuracy_score, classification_report

results = spark.sql(f"""                 
    SELECT 
      i.order_id,
      i.category_level_1 AS actual_level_1,
      i.category_level_2 AS actual_level_2,
      c.pred_level_1,
      c.pred_level_2
    FROM {catalog}.{schema}.{cat_optimize_table} c
    LEFT JOIN {catalog}.{schema}.{invoices_table} i
    ON c.order_id = i.order_id
    WHERE c.invoice_info IS NOT NULL
""").toPandas()

level1_acc = accuracy_score(
  results["actual_level_1"], 
  results["pred_level_1"]
  )
print(f"Level 1 Accuracy: {level1_acc:.3f}")

level2_acc = accuracy_score(
  results["actual_level_2"], 
  results["pred_level_2"]
  )
print(f"Level 2 Accuracy: {level2_acc:.3f}")

print("Level 1 Classification Report:")
print(classification_report(
  results["actual_level_1"], 
  results["pred_level_1"],
  zero_division=0
  ))